# Phân Tích Dữ Liệu Tiêu Thụ Năng Lượng Ngành Thép - EDA

Sổ tay này thực hiện Phân tích Dữ liệu Khám phá (EDA) trên dữ liệu tiêu thụ năng lượng của ngành công nghiệp thép.

**Tổng quan Dataset:**
- 35,040 bản ghi (một năm dữ liệu với khoảng thời gian 15 phút)
- Các đặc trưng: mức tiêu thụ điện, công suất phản kháng, hệ số công suất, lượng khí thải CO2, loại tải
- Phạm vi ngày: 2018-01-01 đến 2018-12-31

In [ ]:
# Thư viện cần thiết
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Cài đặt giao diện biểu đồ
sns.set_theme(style="whitegrid")
%matplotlib inline

---

## 1. Tải Dữ Liệu & Tiền Xử Lý

In [ ]:
# Tải dữ liệu từ file CSV
df = pd.read_csv('Steel_industry_data.csv')

# Chuyển đổi cột 'date' sang định dạng datetime
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y %H:%M')

# Đặt cột 'date' làm index
df.set_index('date', inplace=True)

# Sắp xếp index theo thời gian
df.sort_index(inplace=True)

print(f"Kích thước dataset: {df.shape}")
print(f"Phạm vi ngày: {df.index.min()} đến {df.index.max()}")

---

## 2. Tổng Quan Dữ Liệu

In [ ]:
# Xem thông tin kiểu dữ liệu
print("=== Kiểu Dữ Liệu ===")
df.info()

In [ ]:
# Xem 5 dòng đầu tiên
print("=== 5 Dòng Đầu Tiên ===")
df.head()

---

## 3. Kiểm Tra Giá Trị Null

In [ ]:
# Đếm số giá trị null trong mỗi cột
missing = df.isnull().sum()

# Tính phần trăm giá trị null
missing_pct = (missing / len(df) * 100).round(2)

# Tạo bảng tổng hợp
missing_df = pd.DataFrame({'Số lượng Null': missing, 'Phần trăm Null %': missing_pct})
print("=== Giá Trị Null ===")
print(missing_df)

---

## 4. Thống Kê Mô Tả

In [ ]:
# Thống kê mô tả cho các cột số
print("=== Thống Kê Số ===")
df.describe()

In [ ]:
# Thống kê cho các cột phân loại
print("=== Biến Phân Loại ===")
for col in ['WeekStatus', 'Day_of_week', 'Load_Type']:
    print(f"\n{col}:")
    print(df[col].value_counts())

---

## 5. Kiểm Tra Tính Liên Tục Chuỗi Thời Gian

In [ ]:
# Kiểm tra các timestamp trùng lặp
print(f"Số timestamp trùng lặp: {df.index.duplicated().sum()}")

# Tạo index lý tưởng với khoảng cách 15 phút
perfect_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15min')

# Kiểm tra các khoảng thời gian bị thiếu
missing_intervals = len(perfect_index) - len(df)
print(f"Số khoảng 15 phút bị thiếu: {missing_intervals}")

# Chuyển sang tần suất cố định 15 phút
df = df.asfreq('15min')
print(f"\n=== Sau khi dùng asfreq() ===")
print(f"Giá trị null sau khi tái index:\n{df.isnull().sum().sum()} tổng số giá trị NaN")

---

## 6. Phân Tích Đơn Biến - Phân Phối Biến Mục Tiêu

In [ ]:
# Vẽ histogram và boxplot cho biến mục tiêu
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram với đường KDE
sns.histplot(df['Usage_kWh'], kde=True, bins=50, color='steelblue', ax=axes[0])
axes[0].set_title('Phân Phối Tiêu Thụ Năng Lượng (kWh)')
axes[0].set_xlabel('Mức tiêu thụ (kWh)')
axes[0].set_ylabel('Tần suất')

# Boxplot
sns.boxplot(x=df['Usage_kWh'], ax=axes[1], color='coral')
axes[1].set_title('Biểu Đồ Hộp Tiêu Thụ Năng Lượng (kWh)')
axes[1].set_xlabel('Mức tiêu thụ (kWh)')

plt.tight_layout()
plt.show()

print(f"Usage_kWh: Min={df['Usage_kWh'].min():.2f}, Max={df['Usage_kWh'].max():.2f}, Mean={df['Usage_kWh'].mean():.2f}")

---

## 7. Phân Tích Thời Gian

In [ ]:
# Trích xuất các đặc trưng thời gian từ index
df['Hour'] = df.index.hour           # Giờ trong ngày (0-23)
df['DayOfWeek'] = df.index.dayofweek  # Ngày trong tuần (0-6)
df['Month'] = df.index.month         # Tháng trong năm (1-12)
df['DayName'] = df['DayOfWeek'].map({0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri', 5:'Sat', 6:'Sun'})
print("Đã trích xuất các đặc trưng thời gian: Hour, DayOfWeek, Month, DayName")

In [ ]:
# Biểu đồ mức tiêu thụ trung bình theo giờ trong ngày
plt.figure(figsize=(12, 5))

# Tính trung bình và độ lệch chuẩn theo giờ
hourly_avg = df.groupby('Hour')['Usage_kWh'].agg(['mean', 'std']).reset_index()

# Vẽ đường biểu diễn
sns.lineplot(data=hourly_avg, x='Hour', y='mean', marker='o', color='crimson', linewidth=2)

# Vẽ vùng độ lệch chuẩn
plt.fill_between(hourly_avg['Hour'], hourly_avg['mean'] - hourly_avg['std'], hourly_avg['mean'] + hourly_avg['std'], alpha=0.2, color='crimson')

plt.title('Tiêu Thụ Năng Lượng Trung Bình Theo Giờ Trong Ngày')
plt.xlabel('Giờ trong ngày (0-23)')
plt.ylabel('Tiêu thụ trung bình (kWh)')
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()

In [ ]:
# Biểu đồ mức tiêu thụ trung bình theo ngày trong tuần
plt.figure(figsize=(10, 5))

day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
daily_avg = df.groupby('DayName')['Usage_kWh'].mean().reindex(day_order)

plt.bar(day_order, daily_avg.values, color='teal')
plt.title('Tiêu Thụ Năng Lượng Trung Bình Theo Ngày Trong Tuần')
plt.xlabel('Ngày trong tuần')
plt.ylabel('Tiêu thụ trung bình (kWh)')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Heatmap: Tiêu thụ theo ngày và giờ
hourly_daily_pivot = df.pivot_table(index='DayName', columns='Hour', values='Usage_kWh', aggfunc='mean')
hourly_daily_pivot = hourly_daily_pivot.reindex(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])

plt.figure(figsize=(14, 6))
sns.heatmap(hourly_daily_pivot, cmap='YlOrRd', linewidths=0.5)
plt.title('Heatmap: Tiêu Thụ Trung Bình Theo Ngày và Giờ')
plt.xlabel('Giờ trong ngày')
plt.ylabel('Ngày trong tuần')
plt.show()

---

## 8. Phân Tích Phân Loại - Loại Tải

In [ ]:
# Biểu đồ phân bố và so sánh tiêu thụ theo loại tải
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Đếm số lượng mỗi loại tải
load_counts = df['Load_Type'].value_counts()
axes[0].bar(load_counts.index, load_counts.values, color=['lightgreen', 'gold', 'salmon'])
axes[0].set_title('Phân Bố Các Loại Tải')
axes[0].set_xlabel('Loại tải')
axes[0].set_ylabel('Số lượng')

# Boxplot so sánh tiêu thụ theo loại tải
load_order = ['Light_Load', 'Medium_Load', 'Maximum_Load']
sns.boxplot(data=df, x='Load_Type', y='Usage_kWh', order=load_order, ax=axes[1], palette='Set2')
axes[1].set_title('Tiêu Thụ Năng Lượng Theo Loại Tải')
axes[1].set_xlabel('Loại tải')
axes[1].set_ylabel('Tiêu thụ (kWh)')

plt.tight_layout()
plt.show()

print("\n=== Thống Kê Tiêu Thụ Theo Loại Tải ===")
print(df.groupby('Load_Type')['Usage_kWh'].describe()[['mean', 'min', 'max']])

---

## 9. Phân Tích Trạng Thái Tuần

In [ ]:
# So sánh tiêu thụ giữa ngày trong tuần và cuối tuần
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='WeekStatus', y='Usage_kWh', palette='coolwarm')
plt.title('Tiêu Thụ Năng Lượng: Ngày Trong Tuần vs Cuối Tuần')
plt.xlabel('Trạng thái tuần')
plt.ylabel('Tiêu thụ (kWh)')
plt.show()

print(df.groupby('WeekStatus')['Usage_kWh'].agg(['mean', 'std', 'count']))

---

## 10. Ma Trận Tương Quan

In [ ]:
# Chọn các cột số để tính tương quan
numerical_cols = ['Usage_kWh', 'Lagging_Current_Reactive.Power_kVarh', 
                  'Leading_Current_Reactive_Power_kVarh', 'CO2(tCO2)', 
                  'Lagging_Current_Power_Factor', 'Leading_Current_Power_Factor']

# Tính ma trận tương quan
corr_matrix = df[numerical_cols].corr()

# Vẽ heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, center=0)
plt.title('Ma Trận Tương Quan Các Đặc Trưng Số')
plt.show()

print("Tương Quan Quan Trọng với Usage_kWh:")
print(corr_matrix['Usage_kWh'].sort_values(ascending=False))

---

## 11. Phân Tích Chuỗi Thời Gian - Tính Dừng & Tự Tương Quan

In [ ]:
# Kiểm tra tính dừng bằng Augmented Dickey-Fuller Test
print("=== Kiểm Tra Augmented Dickey-Fuller ===")
adf_result = adfuller(df['Usage_kWh'].dropna())
print(f"Thống kê ADF: {adf_result[0]:.4f}")
print(f"Giá trị p: {adf_result[1]:.6f}")

if adf_result[1] < 0.05:
    print("Kết luận: Chuỗi thời gian LÀ DỪNG")
else:
    print("Kết luận: Chuỗi thời gian KHÔNG DỪNG")

In [ ]:
# Vẽ ACF và PACF
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ACF - Tự tương quan
plot_acf(df['Usage_kWh'].dropna(), lags=96, ax=axes[0])
axes[0].set_title('ACF - Đến 24 Giờ')

# PACF - Tự tương quan riêng phần
plot_pacf(df['Usage_kWh'].dropna(), lags=96, ax=axes[1])
axes[1].set_title('PACF - Đến 24 Giờ')

plt.tight_layout()
plt.show()

print("ACF cho thấy tính chu kỳ 24 giờ mạnh (lag 96). PACF cho thấy các đỉnh ý nghĩa ở lags 1-2.")

---

## 12. Tạo Đặc Trưng

In [ ]:
# Tạo các đặc trưng lag dựa trên PACF
df['Usage_Lag_1'] = df['Usage_kWh'].shift(1)   # 15 phút trước
df['Usage_Lag_2'] = df['Usage_kWh'].shift(2)   # 30 phút trước

# Tạo lag theo mùa (24 giờ = 96 khoảng 15 phút)
df['Usage_Lag_96'] = df['Usage_kWh'].shift(96) # 24 giờ trước

# Tạo các đặc trưng trung bình trượt
df['Rolling_Mean_1h'] = df['Usage_kWh'].shift(1).rolling(window=4).mean()   # 1 giờ qua
df['Rolling_Mean_24h'] = df['Usage_kWh'].shift(1).rolling(window=96).mean() # 24 giờ qua

# Loại bỏ các dòng có NaN do shifting
df_model = df.dropna().copy()

print(f"Kích thước ban đầu: {df.shape}")
print(f"Sau khi tạo đặc trưng: {df_model.shape}")
print("Đặc trưng mới: Usage_Lag_1, Usage_Lag_2, Usage_Lag_96, Rolling_Mean_1h, Rolling_Mean_24h")

---

## 13. Chia Dữ Liệu Train/Test

In [ ]:
# Chia dữ liệu theo thời gian (80% train, 20% test)
split_index = int(len(df_model) * 0.8)

train_df = df_model.iloc[:split_index]
test_df = df_model.iloc[split_index:]

print(f"Train: {train_df.index.min()} đến {train_df.index.max()}")
print(f"Test: {test_df.index.min()} đến {test_df.index.max()}")
print(f"Train: {len(train_df)} mẫu (80%), Test: {len(test_df)} mẫu (20%)")

In [ ]:
# Trực quan hóa phân chia train/test
plt.figure(figsize=(15, 5))
plt.plot(train_df.index, train_df['Usage_kWh'], label='Dữ liệu Train', color='steelblue', alpha=0.8)
plt.plot(test_df.index, test_df['Usage_kWh'], label='Dữ liệu Test', color='orange', alpha=0.8)
plt.axvline(x=train_df.index[-1], color='red', linestyle='--', label='Điểm Chia Train/Test')
plt.title('Chia Dữ Liệu Train/Test Theo Thời Gian (80/20)')
plt.xlabel('Ngày')
plt.ylabel('Tiêu thụ (kWh)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---

## 14. Tóm Tắt Các Phát Hiện Chính

In [ ]:
# Tóm tắt các phát hiện chính
msg = ""
msg += "=== CÁC PHÁT HIỆN CHÍNH ===\n\n"
msg += "1. CHẤT LƯỢNG DỮ LIỆU:\n"
msg += "   - Không có giá trị null, không có khoảng trống trong chuỗi thời gian\n"
msg += "   - 35,040 bản ghi bao phủ toàn bộ năm 2018\n\n"
msg += "2. BIẾN MỤC TIÊU (Usage_kWh):\n"
msg += f"   - Trung bình: {df['Usage_kWh'].mean():.2f} kWh, Max: {df['Usage_kWh'].max():.2f} kWh\n"
msg += "   - Phân phối lệch phải mạnh\n\n"
msg += "3. MẪU THỜI GIAN:\n"
msg += "   - Hàng ngày: Đỉnh tiêu thụ trong 8h - 18h (giờ làm việc)\n"
msg += "   - Hàng tuần: Ngày trong tuần (T2-T4) có tiêu thụ cao nhất\n"
msg += "   - ACF cho thấy tính chu kỳ 24 giờ mạnh\n\n"
msg += "4. BIẾN PHÂN LOẠI:\n"
msg += "   - Maximum_Load có mức tiêu thụ trung bình cao nhất\n"
msg += "   - Ngày trong tuần có tiêu thụ cao gấp ~2 lần cuối tuần\n\n"
msg += "5. TƯƠNG QUAN:\n"
msg += "   - Tương quan dương mạnh với CO2 và công suất phản kháng\n"
msg += "   - Tương quan âm với hệ số công suất\n\n"
msg += "6. TÍNH DỪNG:\n"
msg += "   - Kiểm tra ADF xác nhận chuỗi dừng\n\n"
msg += "7. CHO MÔ HÌNH HÓA:\n"
msg += "   - Tạo đặc trưng lag (Lags 1, 2, 96)\n"
msg += "   - Tạo trung bình trượt (1h, 24h)\n"
msg += "   - Áp dụng chia train/test 80/20 theo thời gian\n"
print(msg)

---

## Các Bước Tiếp Theo

- Xây dựng mô hình dự báo (ARIMA, LSTM, hoặc XGBoost)
- Đánh giá hiệu suất mô hình trên tập test
- Xem xét các đặc trưng bổ sung (ngày lễ, sự kiện đặc biệt)